### Working with Documents in Python

In [ ]:
# !pip install langchain-community
# !pip install unstructured
# !pip install pdfminer
# !pip install --upgrade "unstructured[pdf]"

##### Embedding refers to the process of converting complex data (like words or categories) into dense numerical vectors that capture their meaning and relationships.

In [ ]:
import os
from langchain_community.document_loaders import (
    UnstructuredPDFLoader
)

docs_folder = "./Docs"
documents = []

for filename in os.listdir(docs_folder):
    filepath = os.path.join(docs_folder, filename)
    loader = UnstructuredPDFLoader(filepath)
    documents.extend(loader.load())


Chunking the text into smaller parts can be useful for processing large documents. Here's how you can do it using Python:

```python

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"------ Chuck {i + 1} ------ ")
    print(chunk.page_content)
    print("------------------------------")


In [ ]:
!pip install chromadb
# cmd - ollama pull nomic-embed-text

### Embedding Documents

In [ ]:
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

#- Chroma: A vector database used to store and search embeddings efficiently

embedding = OllamaEmbeddings(model="nomic-embed-text") # - Initializes the embedding model that converts text chunks into numerical vectors.
db = Chroma.from_documents(chunks, embedding, persist_directory="qa_db")

#- Converts the chunks (split documents) into embeddings.
#- Stores them in a Chroma vector database located at "qa_db" for persistent access. New directory will be created with data in same folder. 

retriever = db.as_retriever()
#- Converts the Chroma database into a retriever object.
#- This allows you to query the database using natural language and retrieve relevant chunks based on semantic similarity.

In [ ]:
embedding

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(                       # instance of ChatOllama class is created
    base_url="http://localhost:11434",  # By default, Ollama runs locally on port 11434.
    model="qwen2.5:latest",
    temperature=0.5
)

# - temperature=0.5
# - Controls randomness in the model’s responses.
# - It affects how the model chooses words when multiple options are possible.
# - Lower values (close to 0) → more deterministic, focused answers.
# - If you ask the same question twice, you’ll likely get nearly identical answers.
# - Good for tasks where accuracy and reliability matter
# - Higher values (close to 1) → more creative, varied outputs.
# - Responses vary more between runs.
# - 0.5 is a balanced middle ground.


#### Retrieval QA 

RetrievalQA is a powerful chain in LangChain that combines two key components:
RetrievalQA = Retriever + Language Model (LLM)
It enables a system to:
- Retrieve relevant information from a document store (like a vector database).
- Answer a user’s question using that information via a language model.

- Retriver: Finds the most relevant document chunks based on the user's question
- LLM: Reads those chunks and generates a natural-language answer


In [ ]:
from langchain_classic.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(llm = llm, retriever=retriever)

# - The above line creates a QA pipeline that first retrieves relevant context and then uses the LLM to answer the question.

question = "Explain the system architecture of Netflix"
print(qa_chain.run(question))
# - The question is passed to the QA chain.
# - The retriever fetches relevant chunks from the Chroma vector store.
# - The LLM uses those chunks to generate a detailed answer.


# RetrievalQA effectively integrates document retrieval with the advanced question-answering capabilities of language models, enhancing the ability to provide accurate and relevant information from large datasets. This synergy allows users to efficiently access information while also obtaining direct answers to their queries.
# Vector stores measure document similarity by calculating the distances between their vector embeddings in high-dimensional space. This method captures the semantic relationships between documents, allowing for more effective retrieval of relevant information.